# 07 - RQ3 on test, once
Every model must be in selection.lock.json. Paired bootstrap vs TF-IDF/LR, Holm, DeLong.

In [ ]:
# Colab setup: repository at /content/kev-text (git clone or unzip), then run all.
import os, sys
REPO = "/content/kev-text"
if os.path.exists(REPO):
    os.chdir(REPO)
sys.path.insert(0, os.getcwd())
# !pip install -q -r requirements.txt          # once per Colab session
import numpy as np, pandas as pd
from src import config, io_utils, labels, evaluate
SYN = False   # True = dry run on src.synthetic; everything goes to outputs_synthetic/

In [ ]:
if SYN:
    from src import synthetic
    df = synthetic.make(6000)
    if not config.SPLITS_LOCK.exists():                   # mirrors the extras notebook 02 writes
        from src import sensitivity
        sp0 = labels.assign_split(df["published_date"])
        m = sensitivity.kev_retroactive_monthly(df[["cve_id", "kev_date_added"]].dropna(),
                                                df[["cve_id", "published_date"]])
        vb = df[(sp0 == "val") & (df["published_date"] > config.VAL_A_END)]
        labels.write_lock(df, "initial_seal", "synthetic dry run",
                          extra={"s1": sensitivity.s1_record(m, df, sp0),
                                 "conformal_alpha": config.conformal_alpha(int(vb["label_12m"].sum()))})
else:
    df = pd.read_parquet(config.DATA / "dataset_labelled.parquet")
lock = labels.require_lock()
X, y, split = labels.get_xy(df, labels.CLEAN)
split.value_counts()

In [ ]:
from src import validate
yt = y[split == "test"]
ref = evaluate.load_scores("tfidf-lr", "test", is_synthetic=SYN).loc[yt.index]
rows = []
for model in evaluate.list_models(is_synthetic=SYN):
    s = evaluate.load_scores(model, "test", is_synthetic=SYN).reindex(yt.index)
    ok = s.notna()
    m = evaluate.test_metrics(model, yt[ok], s[ok], is_synthetic=SYN)
    if model != "tfidf-lr":
        pb = evaluate.paired_bootstrap(yt[ok], s[ok], ref[ok])
        m.update(delta_vs_tfidf=pb["delta"], lo=pb["lo"], hi=pb["hi"], p_boot=pb["p"],
                 p_delong=evaluate.delong(yt[ok], s[ok], ref[ok])["p"])
    rows.append({"model": model, **m})
t = pd.DataFrame(rows).set_index("model")
t["p_holm"] = validate.holm(t["p_boot"])
evaluate.save_table(t, "table_rq3", is_synthetic=SYN)
t.sort_values("pr_auc", ascending=False)